# Fresh AI: Fresh-knowledge fine-tune of `fresh-unified-1` (run 4)

Stage 2. Starts from the run 3 checkpoint and teaches Fresh facts (docs Q&A, expanded, intake and beyond-SI rows) while replaying general Swahili/English data so nothing is forgotten. It measures **before and after** on a held-out set of 420 Fresh questions that are never trained on.

## Setup (once)
1. **Accelerator:** GPU (T4). **Internet:** On.
2. **Add run 3's output as an input:** *Add Input → Your Work (or Notebook Output Files) → your run 3 notebook*. It must contain `fresh-unified-1-run3.pt` and `tokenizer-v2.json` (the fine-tune must reuse the same tokenizer).
3. *Save Version → Save & Run All (Commit)*. Takes about 1 to 2 hours.

It stops at the first problem. **Honest expectation:** a 50M model will learn to recall Fresh doc sections better, not to reason. The checkpoint stays `training-required` and is never production-ready.

In [ ]:
# 1. Setup
import subprocess, re, os, sys, glob, json
W = "/kaggle/working"
def run(cmd, check=True):
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    lines = []
    for line in p.stdout:
        print(line, end=""); lines.append(line)
    p.wait()
    if check and p.returncode:
        raise RuntimeError(f"command failed ({p.returncode}): {cmd}")
    return p.returncode, "".join(lines)

rc, _ = run("nvidia-smi -L", check=False)
if rc: raise RuntimeError("No GPU: set Accelerator to GPU in the notebook settings.")
if not os.path.exists(f"{W}/fresh-web-lite"):
    run(f"git clone --depth 1 https://github.com/kanarafat54-boop/fresh-web-lite.git {W}/fresh-web-lite")
else:
    run(f"git -C {W}/fresh-web-lite pull --ff-only")
os.chdir(f"{W}/fresh-web-lite")
run("pip -q install tokenizers numpy")
import torch; print("torch", torch.__version__, "cuda", torch.cuda.is_available())

def find(name):
    hits = glob.glob(f"/kaggle/input/**/{name}", recursive=True)
    if not hits: raise RuntimeError(f"{name} not found under /kaggle/input. Add the run 3 notebook output as an input (see the top of this notebook).")
    return hits[0]
CKPT = find("fresh-unified-1-run3.pt")
TOK = find("tokenizer-v2.json")
print("checkpoint:", CKPT); print("tokenizer:", TOK)
for f in ("data/fresh-eval/fresh-knowledge-recall-v1.jsonl", "data/fresh-training/fresh-docs-qa-v1.jsonl", "training/mix_recipe_sft_v1.json"):
    if not os.path.exists(f): raise RuntimeError(f"{f} missing: the repo branch with the fine-tune work is not merged yet.")

In [ ]:
# 2. Data. Rebuild the run 3 mix (only for its fixed validation split, so before/after is comparable),
#    build the fine-tune mix, and encode BOTH with the run 3 tokenizer (never retrain the tokenizer).
D = f"{W}/data"; os.makedirs(D, exist_ok=True)
if not os.path.exists(f"{D}/mix/validation.jsonl.gz"):
    run(f"python3 training/build_training_mix.py --recipe training/mix_recipe_v1.json --out-dir {D}/mix")
if not os.path.exists(f"{D}/sft/train.jsonl.gz"):
    run(f"python3 training/build_training_mix.py --recipe training/mix_recipe_sft_v1.json --out-dir {D}/sft")
for split in ("train", "validation"):
    run(f"python3 training/fresh_tokenizer_v2.py encode --tokenizer {TOK} --input {D}/sft/{split}.jsonl.gz --output {D}/sft-{split}.bin")
run(f"ls -lh {D} {D}/sft")

In [ ]:
# 3. BEFORE: evaluate run 3 on the held-out Fresh questions and per-group perplexity.
EV = (f"python3 training/eval_fresh_model.py --tokenizer {TOK} --spec training/model_spec.json "
      f"--recall-eval data/fresh-eval/fresh-knowledge-recall-v1.jsonl --validation {D}/mix/validation.jsonl.gz --limit-recall 150")
run(f"{EV} --checkpoint {CKPT} --label run3-before --out {W}/eval-before.json")

In [ ]:
# 4. Smoke test (tiny model, 20 steps, compile fallback).
base = (f"python3 training/train_fresh_unified_full.py --train-bin {D}/sft-train.bin --val-bin {D}/sft-validation.bin --tokenizer {TOK} "
        f"--spec training/model_spec_smoke.json --seq-len 256 --micro-batch 4 --accum 1 --max-steps 20 --warmup 5 --eval-every 10 --save-every 20 "
        f"--eval-blocks 8 --out-dir {W}/smoke --run-id smoke")
COMPILE = " --compile"
run(f"rm -rf {W}/smoke")
rc, out = run(base + COMPILE, check=False)
if rc:
    print("\n--compile failed; retrying without it"); COMPILE = ""
    run(f"rm -rf {W}/smoke")
    rc, out = run(base, check=False)
if rc: raise RuntimeError("Smoke test failed even without --compile. Send the error to Claude.")
print("SMOKE OK. compile flag:", COMPILE.strip() or "off")

In [ ]:
# 5. FINE-TUNE from run 3 (low learning rate so general ability is kept). Resumable every 50 steps.
cmd = (f"python3 training/train_fresh_unified_full.py --train-bin {D}/sft-train.bin --val-bin {D}/sft-validation.bin --tokenizer {TOK} "
       f"--spec training/model_spec.json --seq-len 1024 --micro-batch 8 --accum 4 --epochs 3 --lr 1e-4 --warmup 20 --save-every 50 --eval-every 100{COMPILE} "
       f"--init-from {CKPT} --mix-manifest {D}/sft/manifest.json --out-dir {W}/train-run4 --run-id fresh-unified-1-run4")
run(cmd)

In [ ]:
# 6. AFTER: same evaluation on the fine-tuned checkpoint, then the comparison. Send this output to Claude.
run(f"{EV} --checkpoint {W}/train-run4/fresh-unified-1-run4.pt --label run4-after --out {W}/eval-after.json")
b = json.load(open(f"{W}/eval-before.json")); a = json.load(open(f"{W}/eval-after.json"))
print("BEFORE:", json.dumps(b.get("recall", b), indent=1)[:1500])
print("AFTER: ", json.dumps(a.get("recall", a), indent=1)[:1500])
run(f"cat {W}/train-run4/fresh-unified-1-run4.manifest.json; cat {W}/train-run4/fresh-unified-1-run4.samples.txt")
run(f"rm -f {W}/train-run4/last.pt; cp {D}/sft/manifest.json {W}/train-run4/sft-mix-manifest.json; cp {W}/eval-before.json {W}/eval-after.json {W}/train-run4/; ls -lh {W}/train-run4")